In [0]:
from pyspark.sql.functions import *

In [0]:
# ✅ 표시 기준 (저장은 여전히 UTC) 
spark.conf.set("spark.sql.session.timeZone", "Asia/Seoul")

In [0]:

fact_df = spark.table(
    "hive_metastore.demo_airstatus_silver.SLV_temp_fact_air_quality_6h"
)


In [0]:
display(fact_df[fact_df['stationName']=="중구"])

stationName,dataTime,khaiValue,khaiGrade,pm10Value,pm25Value,year,month,day,hour
중구,2026-05-08T06:00:00+09:00,54,2,19,9,2026,5,8,6
중구,2026-05-08T04:00:00+09:00,58,2,22,13,2026,5,8,4
중구,2026-05-08T02:00:00+09:00,57,2,28,18,2026,5,8,2
중구,2026-05-08T03:00:00+09:00,58,2,25,15,2026,5,8,3
중구,2026-05-08T01:00:00+09:00,56,2,28,22,2026,5,8,1
중구,2026-05-08T05:00:00+09:00,57,2,19,8,2026,5,8,5


In [0]:
from pyspark.sql.types import DoubleType

stations_df = spark.table(
    "hive_metastore.demo_airstatus_bronze.BRZ_seoul_stations"
).select(
    "stationName",
    col("dmX").cast(DoubleType()),
    col("dmY").cast(DoubleType()),
    "addr"
)

In [0]:

fact_geo_df = (
    fact_df
    .join(
        stations_df,
        on="stationName",
        how="left"
    )
)


In [0]:
fact_geo_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable(
        "hive_metastore.demo_airstatus_gold.GLD_fact_air_quality_dashboard"
    )